# Fresh confirmatory input-domain review — statement-only

This notebook validates the frozen 30-task, 60-candidate confirmatory trigger population using only task specifications, statement-visible input examples, and the raw generated input strings. It does not load candidate code, labels, reference solutions, provided outputs, secret/non-visible examples, saved model calls, or execution outcomes; it never executes a candidate or calls a model/network service.

Every validator is task-specific and checks the complete stated input syntax, bounds, cardinalities, and semantic promises needed to establish input-domain membership. Numeric fields allow ordinary token whitespace. Original record positions and input indexes are preserved with SHA-256 hashes. The reviewed-input run is a transparent, order-preserving view containing only inputs classified `valid`; any unresolved input, failed source record, missing candidate, or candidate with zero valid inputs fails closed before that view is written. Existing artifacts must match byte-for-byte, making default Restart-and-Run-All offline and immutable.


In [1]:
from pathlib import Path
from collections import Counter, deque
from io import BytesIO
import hashlib, ijson, json, os, re, sys

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
assert Path.cwd() == REPO and (REPO / "data").is_dir(), f"not at repository root: {Path.cwd()}"
DATA = Path("data/azure-terra-pbt-fresh30-second-revision-s300-v1.json")
SOURCE_RUN = "azure-terra-pbt-fresh30-second-revision-s300-v1-triggers"
REVIEWED_RUN = "azure-terra-pbt-fresh30-second-revision-s300-v1-reviewed-inputs"
NOTEBOOK = Path("notebooks/azure_pbt_confirmatory_domain_review.ipynb")
SOURCE_DIR = Path("runs") / SOURCE_RUN
REVIEWED_DIR = Path("runs") / REVIEWED_RUN
REVIEW_PATH = SOURCE_DIR / "domain-review-v1.json"

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

def file_sha256(path):
    return sha256_bytes(path.read_bytes())

def json_bytes(document):
    return (json.dumps(document, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8")

def immutable_bytes(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_bytes() == payload, f"immutable artifact differs: {path}"
    else:
        with path.open("xb") as handle:
            handle.write(payload)

def notebook_source_sha256(path):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source_only = [{"cell_type": cell["cell_type"], "source": cell["source"]} for cell in notebook["cells"]]
    canonical = json.dumps(source_only, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return sha256_bytes(canonical)

def projected_dataset(path):
    def values(prefix):
        with path.open("rb") as handle:
            return list(ijson.items(handle, prefix))
    task_ids = values("tasks.item.task_id")
    specifications = values("tasks.item.specification")
    visible_indexes = [tuple(int(index) for index in indexes) for indexes in values("tasks.item.statement_visible")]
    candidates = []
    visible_inputs = [[] for _ in task_ids]
    task_index = -1
    input_index = None
    with path.open("rb") as handle:
        for prefix, event, value in ijson.parse(handle):
            if prefix == "tasks.item" and event == "start_map":
                task_index += 1; candidates.append([])
            elif prefix == "tasks.item.candidates.item.candidate_id" and event in ("string", "number"):
                candidates[task_index].append(str(value))
            elif prefix == "tasks.item.provided_inputs" and event == "start_array":
                input_index = 0
            elif prefix == "tasks.item.provided_inputs.item" and event == "string":
                if input_index in visible_indexes[task_index]:
                    visible_inputs[task_index].append(value)
                input_index += 1
            elif prefix == "tasks.item.provided_inputs" and event == "end_array":
                input_index = None
    assert len(task_ids) == len(specifications) == len(visible_indexes) == len(candidates) == 30
    assert all(len(group) == 2 for group in candidates)
    assert all(len(found) == len(indexes) for found, indexes in zip(visible_inputs, visible_indexes))
    return {task_id: {"specification": specification, "visible_indexes": indexes,
                      "visible_inputs": tuple(examples), "candidate_ids": tuple(candidate_group)}
            for task_id, specification, indexes, examples, candidate_group
            in zip(task_ids, specifications, visible_indexes, visible_inputs, candidates)}

SAFE_RECORD_FIELDS = {"run_name", "protocol", "split", "task_id", "candidate_id",
                      "failed", "blame", "reason", "n_requested", "n_parsed", "dropped"}
def projected_records(path):
    records = []
    for record_index, raw_with_newline in enumerate(path.read_bytes().splitlines(keepends=True)):
        raw = raw_with_newline.rstrip(b"\r\n")
        if not raw.strip():
            continue
        row = {"source_record_index": record_index, "source_record_sha256": sha256_bytes(raw), "inputs": []}
        for prefix, event, value in ijson.parse(BytesIO(raw)):
            if prefix in SAFE_RECORD_FIELDS and event in ("string", "number", "boolean", "null"):
                row[prefix] = value
            elif prefix == "inputs.item" and event in ("string", "number", "boolean", "null"):
                row["inputs"].append(value)
        records.append(row)
    return records

dataset = projected_dataset(DATA)
source_records_path = SOURCE_DIR / "records.jsonl"
source_config_path = SOURCE_DIR / "config.json"
rows = projected_records(source_records_path)
assert len(rows) == 60 and len({row["candidate_id"] for row in rows}) == 60
assert all(row["failed"] is False for row in rows)
assert all(row["split"] == "test" and row["n_parsed"] == 10 and row["dropped"] == 0 for row in rows)
assert sum(len(row["inputs"]) for row in rows) == 600 and all(len(row["inputs"]) == 10 for row in rows)
expected_candidate_ids = {candidate_id for task in dataset.values() for candidate_id in task["candidate_ids"]}
assert {row["candidate_id"] for row in rows} == expected_candidate_ids


In [2]:
class DomainInvalid(ValueError):
    pass

def require(condition, reason):
    if not condition:
        raise DomainInvalid(reason)

class Tokens:
    def __init__(self, stdin):
        require(isinstance(stdin, str), "stdio input must be a string")
        self.values = stdin.split(); self.position = 0
    def word(self):
        require(self.position < len(self.values), "missing input token")
        value = self.values[self.position]; self.position += 1; return value
    def integer(self, low, high, field="integer"):
        token = self.word()
        require(re.fullmatch(r"[+-]?[0-9]+", token) is not None, f"{field} is not an integer token")
        value = int(token)
        require(low <= value <= high, f"{field} outside {low}..{high}")
        return value
    def vector(self, count, low, high, field="value"):
        return [self.integer(low, high, field) for _ in range(count)]
    def end(self):
        require(self.position == len(self.values), "extra input tokens")

def modular_shift_exists(a, b, modulus):
    target = Counter(b)
    anchor = a[0]
    return any(Counter((value + (candidate - anchor)) % modulus for value in a) == target for candidate in target)

def validate(task_id, stdin):
    try:
        t = Tokens(stdin)
        if task_id == "1175":
            left = t.integer(1, 10**18, "L"); right = t.integer(1, 10**18, "R"); require(left <= right, "L exceeds R")
        elif task_id == "1223":
            n = t.integer(2, 10**5, "N"); p = t.vector(n, 1, n, "P_i"); require(set(p) == set(range(1, n + 1)), "P is not a permutation of 1..N")
        elif task_id == "1361":
            n = t.integer(3, 100, "n"); a = t.vector(n, 1, 1000, "a_i"); require(all(x < y for x, y in zip(a, a[1:])), "hold heights are not strictly increasing")
        elif task_id == "1383":
            n = t.integer(1, 2000, "n"); modulus = t.integer(1, 10**9, "m")
            a = t.vector(n, 0, modulus - 1, "a_i"); b = t.vector(n, 0, modulus - 1, "b_i")
            require(modular_shift_exists(a, b, modulus), "no guaranteed modular shift maps multiset a to b")
        elif task_id in ("1433", "838"):
            high = 1000 if task_id == "1433" else 50
            n = t.integer(1, high, "n"); m = t.integer(1, high, "m"); cells = t.vector(n * m, 0, 1, "cell")
            if task_id == "1433": require(1 in cells, "stage plan contains no actor")
        elif task_id == "1553":
            n = t.integer(1, 1000, "n"); height = t.integer(1, 10**9, "h"); t.vector(n, 1, height, "a_i")
        elif task_id == "1670":
            home, away = t.word(), t.word()
            require(re.fullmatch(r"[A-Z]{1,20}", home) is not None and re.fullmatch(r"[A-Z]{1,20}", away) is not None, "team name format")
            require(home != away, "team names are not distinct")
            n = t.integer(1, 90, "n"); minutes = []
            for _ in range(n):
                minutes.append(t.integer(1, 90, "minute")); require(t.word() in ("h", "a"), "team code must be h or a")
                t.integer(1, 99, "player number"); require(t.word() in ("y", "r"), "card code must be y or r")
            require(all(x < y for x, y in zip(minutes, minutes[1:])), "fouls are not strictly chronological")
        elif task_id == "1681":
            owned, wanted = t.word(), t.word()
            require(re.fullmatch(r"[a-z]{1,1000}", owned) is not None, "owned color string format")
            require(re.fullmatch(r"[a-z]{1,1000}", wanted) is not None, "wanted color string format")
        elif task_id == "1737":
            n = t.integer(1, 1000, "n"); projects = []
            for _ in range(n):
                name = t.word(); require(re.fullmatch(r"[a-z]{1,10}", name) is not None, "project name format")
                version = t.integer(1, 10**6, "project version"); count = t.integer(0, n - 1, "dependency count"); dependencies = []
                for _ in range(count):
                    dependency_name = t.word(); require(re.fullmatch(r"[a-z]{1,10}", dependency_name) is not None, "dependency name format")
                    dependencies.append((dependency_name, t.integer(1, 10**6, "dependency version")))
                projects.append(((name, version), dependencies))
            identities = [identity for identity, _ in projects]; require(len(set(identities)) == n, "duplicate project description")
            described = set(identities); graph = {identity: {dependency for dependency in dependencies if dependency in described} for identity, dependencies in projects}
            indegree = Counter(dependency for dependencies in graph.values() for dependency in dependencies)
            ready = deque(identity for identity in identities if indegree[identity] == 0); visited = 0
            while ready:
                identity = ready.popleft(); visited += 1
                for dependency in graph[identity]:
                    indegree[dependency] -= 1
                    if indegree[dependency] == 0: ready.append(dependency)
            require(visited == n, "project dependencies contain a cycle")
        elif task_id == "1743":
            n = t.integer(1, 3000, "n"); t.vector(3 * n, 0, 10**5, "joy")
        elif task_id in ("1766", "1864", "2046"):
            maxima = {"1766": (1000, 1000), "1864": (1000, 10**6), "2046": (100000, None)}
            n_high, value_high = maxima[task_id]; n = t.integer(1, n_high, "n"); values = t.vector(n, 1, n if value_high is None else value_high, "value")
            require(len(set(values)) == n, "values are not distinct")
            if task_id == "2046": require(set(values) == set(range(1, n + 1)), "snack sizes are not a permutation of 1..n")
        elif task_id == "1941":
            t.integer(1, 10**6, "A"); t.integer(1, 10**6, "B"); n = t.integer(1, 10**5, "n"); t.vector(3 * n, 1, 10**6, "query value")
        elif task_id == "1958":
            n = t.integer(1, 40, "n"); price = t.integer(2, 1000, "p"); require(price % 2 == 0, "p is not even")
            buyers = [t.word() for _ in range(n)]; require(all(value in ("half", "halfplus") for value in buyers), "buyer description")
            require(buyers[-1] == "halfplus", "guaranteed positive final sale requires last buyer halfplus")
        elif task_id == "2183":
            a = t.integer(1, 3, "a"); b = t.integer(1, 3, "b"); require(a != b, "brother numbers are not distinct")
        elif task_id == "2222":
            n = t.integer(2, 3 * 10**5, "n"); t.vector(n, 0, 1, "operation")
            for node in range(2, n + 1): t.integer(1, node - 1, f"parent of {node}")
        elif task_id == "4089":
            t.integer(1, 1000000000000001, "N")
        elif task_id == "4365":
            t.integer(2, 100, "K")
        elif task_id == "4414":
            q = t.integer(1, 10**4, "q"); t.vector(4 * q, 1, 10**9, "test-case value")
        elif task_id == "4442":
            t.integer(1, 9, "a"); t.integer(1, 9, "b")
        elif task_id == "570":
            t.integer(1, 10**9, "a"); t.integer(1, 10**9, "b")
        elif task_id == "630":
            n = t.integer(1, 10**5, "n"); t.integer(0, n, "k")
            for index in range(1, n + 1): t.integer(0, index - 1, f"a_{index}")
        elif task_id == "632":
            count = t.integer(1, 100, "t"); total_n = 0
            for _ in range(count): total_n += t.integer(2, 10**6, "n"); t.integer(1, 10**9, "k")
            require(total_n <= 10**6, "sum of n exceeds 1000000")
        elif task_id == "677":
            q = t.integer(1, 500, "q")
            for _ in range(q):
                left = t.integer(1, 10**9, "l"); right = t.integer(1, 10**9, "r"); t.integer(1, 10**9, "d"); require(left <= right, "l exceeds r")
        elif task_id == "756":
            n = t.integer(1, 90, "n"); minutes = t.vector(n, 1, 90, "t_i"); require(all(x < y for x, y in zip(minutes, minutes[1:])), "interesting minutes are not strictly increasing")
        elif task_id == "842":
            token = t.word(); require(re.fullmatch(r"[+]?[0-9]+", token) is not None, "n is not a positive integer token")
            digits = token.lstrip("+").lstrip("0"); require(bool(digits), "n is below 1")
            require(len(digits) < 100001 or (len(digits) == 100001 and digits == "1" + "0" * 100000), "n exceeds 10^100000")
        elif task_id == "86":
            xp, yp, xv, yv = t.vector(4, 0, 10**5, "coordinate")
            require((xp, yp) != (xv, yv), "pawns start in the same cell"); require((xp, yp) != (0, 0) and (xv, yv) != (0, 0), "a pawn starts at (0,0)")
        elif task_id == "966":
            t.integer(1000, 9000, "year")
        else:
            return {"status": "unresolved", "reason": "no statement-derived validator for task"}
        t.end()
        return {"status": "valid", "reason": "all stated input-domain constraints satisfied"}
    except DomainInvalid as error:
        return {"status": "invalid", "reason": str(error)}
    except Exception as error:
        return {"status": "unresolved", "reason": f"validator exception: {type(error).__name__}: {error}"}

RULES = {
    "1175": "1<=L<=R<=1e18", "1223": "2<=N<=1e5; P permutation 1..N",
    "1361": "3<=n<=100; strictly increasing a in 1..1000", "1383": "n,m bounds; 2n residues; guaranteed feasible common modular shift",
    "1433": "1<=n,m<=1000; n*m bits; at least one actor", "1553": "n,h bounds; n bottle heights in 1..h",
    "1670": "distinct uppercase team names; 1..90 strictly chronological typed foul records", "1681": "two nonempty lowercase strings, each length<=1000",
    "1737": "1..1000 unique project descriptions; names/versions/counts bounded; dependency graph acyclic", "1743": "1<=n<=3000; exactly 3n joy values in 0..1e5",
    "1766": "1<=n<=1000; n distinct card values in 1..1000", "1864": "1<=n<=1000; n distinct banknote values in 1..1e6",
    "1941": "A,B,n bounds; exactly n query triples in 1..1e6", "1958": "n,p bounds; p even; n half/halfplus records; guaranteed positive final sale",
    "2046": "1<=n<=1e5; snack sizes permutation 1..n", "2183": "two distinct brother numbers in 1..3",
    "2222": "n bounds; n binary operations; parent f_i in 1..i-1", "4089": "one integer N in 1..1000000000000001",
    "4365": "one integer K in 2..100", "4414": "1<=q<=1e4; exactly q positive bounded quadruples",
    "4442": "two one-digit positive integers", "570": "two integers in 1..1e9",
    "630": "n,k bounds; exactly n links with 0<=a_i<i", "632": "1<=t<=100; query bounds; sum n<=1e6",
    "677": "1<=q<=500; q triples with 1<=l<=r<=1e9 and bounded d", "756": "1<=n<=90; strictly increasing interesting minutes in 1..90",
    "838": "1<=n,m<=50; exactly n*m binary cells", "842": "one positive integer n<=10^100000",
    "86": "four coordinates in 0..1e5; distinct pawn cells; neither at origin", "966": "one year in 1000..9000"}
assert set(RULES) == set(dataset)


In [3]:
CASES = {
"1175": ("1 1000000000000000000", "2 1"), "1223": ("2 2 1", "2 1 1"),
"1361": ("3 1 2 1000", "3 1 1 2"), "1383": ("2 3 0 1 1 2", "2 3 0 0 0 1"),
"1433": ("1 1 1", "1 1 0"), "1553": ("1 1 1", "1 2 3"),
"1670": ("A B 1 1 h 1 y", "A A 1 1 h 1 y"), "1681": ("a z", "A z"),
"1737": ("1 a 1 0", "1 a 1 1 a 1"), "1743": ("1 0 100000 1", "1 0 -1 1"),
"1766": ("2 1 1000", "2 1 1"), "1864": ("2 1 1000000", "2 1 1"),
"1941": ("1 1000000 1 1 1 1", "1 1 1 1 1 0"), "1958": ("1 2 halfplus", "1 2 half"),
"2046": ("3 3 1 2", "3 1 1 2"), "2183": ("1 3", "2 2"),
"2222": ("2 0 1 1", "2 0 1 2"), "4089": ("1000000000000001", "1000000000000002"),
"4365": ("100", "1"), "4414": ("1 1 1000000000 1 1000000000", "1 1 1 0 1"),
"4442": ("1 9", "0 9"), "570": ("1 1000000000", "0 1"),
"630": ("2 2 0 1", "2 0 0 2"), "632": ("1 1000000 1", "2 500001 1 500000 1"),
"677": ("1 1 1000000000 1", "1 2 1 1"), "756": ("2 1 90", "2 2 2"),
"838": ("1 1 0", "1 1 2"), "842": ("1", "0"),
"86": ("0 1 100000 100000", "1 1 1 1"), "966": ("9000", "9001")
}
assert set(CASES) == set(RULES)
for task_id, (good, bad) in CASES.items():
    assert validate(task_id, good)["status"] == "valid", (task_id, validate(task_id, good))
    assert validate(task_id, bad)["status"] == "invalid", (task_id, validate(task_id, bad))
    assert validate(task_id, "")["status"] == "invalid", task_id
    assert validate(task_id, good + " EXTRA")["status"] == "invalid", task_id
for task_id, task in dataset.items():
    for visible_input in task["visible_inputs"]:
        assert validate(task_id, visible_input)["status"] == "valid", (task_id, validate(task_id, visible_input))
EXTRA = [("1383", "3 2 0 0 0 1 1 1", "valid"), ("1670", "A B 2 2 h 1 y 1 a 1 r", "invalid"),
         ("1737", "2 a 1 1 b 1 b 1 1 a 1", "invalid"), ("1958", "2 10 half halfplus", "valid"),
         ("2222", "3 1 0 1 1 2", "valid"), ("630", "1 0 0", "valid"),
         ("842", "1" + "0" * 100000, "valid"), ("842", "2" + "0" * 100000, "invalid")]
for task_id, value, expected in EXTRA:
    assert validate(task_id, value)["status"] == expected, (task_id, validate(task_id, value))
assert validate("unknown", "1")["status"] == "unresolved"
print({"task_good_bad_pairs": len(CASES), "empty_tests": len(CASES), "extra_token_tests": len(CASES),
       "visible_example_tests": sum(len(task["visible_inputs"]) for task in dataset.values()), "semantic_edge_tests": len(EXTRA), "model_calls": 0})


{'task_good_bad_pairs': 30, 'empty_tests': 30, 'extra_token_tests': 30, 'visible_example_tests': 83, 'semantic_edge_tests': 8, 'model_calls': 0}


In [4]:
totals = Counter(); per_task = {task_id: Counter() for task_id in dataset}; candidate_reviews = {}; issues = []
reviewed_rows = []
for row in rows:
    task_id, candidate_id = row["task_id"], row["candidate_id"]
    require(task_id in dataset, "source record has unknown task")
    checked = []
    for source_input_index, value in enumerate(row["inputs"]):
        decision = validate(task_id, value)
        item = {"source_input_index": source_input_index, "input_sha256": sha256_bytes(value.encode("utf-8")), **decision}
        checked.append(item); totals[decision["status"]] += 1; per_task[task_id][decision["status"]] += 1
        if decision["status"] != "valid": issues.append({"task_id": task_id, "candidate_id": candidate_id, **item})
    valid_indices = [item["source_input_index"] for item in checked if item["status"] == "valid"]
    invalid_indices = [item["source_input_index"] for item in checked if item["status"] == "invalid"]
    unresolved_indices = [item["source_input_index"] for item in checked if item["status"] == "unresolved"]
    candidate_reviews[candidate_id] = {"task_id": task_id, "source_record_index": row["source_record_index"],
        "source_record_sha256": row["source_record_sha256"], "inputs": checked, "valid_indices": valid_indices,
        "invalid_indices": invalid_indices, "unresolved_indices": unresolved_indices,
        "raw_unique_inputs": len(set(row["inputs"])), "whitespace_normalized_unique_inputs": len({" ".join(value.split()) for value in row["inputs"]})}
    reviewed_rows.append({"run_name": REVIEWED_RUN, "protocol": "statement_domain_review", "split": row["split"],
        "task_id": task_id, "candidate_id": candidate_id, "failed": False, "blame": None, "reason": "",
        "inputs": [row["inputs"][index] for index in valid_indices], "n_requested": row["n_requested"],
        "n_parsed": len(valid_indices), "dropped": len(invalid_indices), "source_run": SOURCE_RUN,
        "source_record_index": row["source_record_index"], "source_record_sha256": row["source_record_sha256"],
        "source_input_indices": valid_indices, "source_input_sha256": [checked[index]["input_sha256"] for index in valid_indices]})

unresolved_candidate_ids = sorted(candidate_id for candidate_id, review in candidate_reviews.items() if review["unresolved_indices"])
zero_valid_candidate_ids = sorted(candidate_id for candidate_id, review in candidate_reviews.items() if not review["valid_indices"])
failed_source_candidate_ids = sorted(row["candidate_id"] for row in rows if row["failed"])
all_candidates_resolved = (not unresolved_candidate_ids and not zero_valid_candidate_ids and not failed_source_candidate_ids
                           and set(candidate_reviews) == expected_candidate_ids)
records_payload = b"".join(json.dumps(row, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8") + b"\n" for row in reviewed_rows)
records_sha256 = sha256_bytes(records_payload) if all_candidates_resolved else None
config = {"schema_version": 1, "protocol": "statement_domain_review", "run_name": REVIEWED_RUN,
          "source_run": SOURCE_RUN, "data": str(DATA), "dataset_sha256": file_sha256(DATA),
          "source_config_sha256": file_sha256(source_config_path), "source_records_sha256": file_sha256(source_records_path),
          "review_notebook": str(NOTEBOOK), "review_notebook_source_sha256": notebook_source_sha256(NOTEBOOK),
          "candidate_count": len(reviewed_rows), "input_filter": "retain status=valid at original index, preserving order and bytes"}
config_payload = json_bytes(config)
if all_candidates_resolved:
    immutable_bytes(REVIEWED_DIR / "records.jsonl", records_payload)
    immutable_bytes(REVIEWED_DIR / "config.json", config_payload)
review = {"schema_version": 1, "scope": "30 frozen confirmatory tasks; 60 candidates; 600 generated inputs",
    "method": "statement/specification-visible examples and raw generated inputs only; no candidate/label/reference/output/outcome inspection, execution, model, or network",
    "dataset_sha256": file_sha256(DATA), "source_config_sha256": file_sha256(source_config_path),
    "source_records_sha256": file_sha256(source_records_path), "review_notebook_source_sha256": notebook_source_sha256(NOTEBOOK),
    "reviewed_config_sha256": sha256_bytes(config_payload) if all_candidates_resolved else None,
    "input_records_sha256": records_sha256, "all_candidates_resolved": all_candidates_resolved,
    "totals": {status: totals[status] for status in ("valid", "invalid", "unresolved")},
    "source": {"tasks": len(dataset), "candidates": len(rows), "inputs": sum(len(row["inputs"]) for row in rows),
               "failed_records": len(failed_source_candidate_ids), "parsed": sum(row["n_parsed"] for row in rows), "dropped": sum(row["dropped"] for row in rows)},
    "reviewed": {"candidates": len(reviewed_rows) if all_candidates_resolved else 0, "inputs": totals["valid"] if all_candidates_resolved else 0},
    "unresolved_candidate_ids": unresolved_candidate_ids, "zero_valid_candidate_ids": zero_valid_candidate_ids,
    "failed_source_candidate_ids": failed_source_candidate_ids, "invalid_task_ids": sorted({issue["task_id"] for issue in issues if issue["status"] == "invalid"}, key=int),
    "unresolved_task_ids": sorted({issue["task_id"] for issue in issues if issue["status"] == "unresolved"}, key=int),
    "task_specification_sha256": {task_id: sha256_bytes(task["specification"].encode("utf-8")) for task_id, task in dataset.items()},
    "rules": RULES, "per_task": {task_id: {status: per_task[task_id][status] for status in ("valid", "invalid", "unresolved")} for task_id in dataset},
    "candidates": candidate_reviews, "issues": issues}
immutable_bytes(REVIEW_PATH, json_bytes(review))
assert all_candidates_resolved, {"unresolved_candidate_ids": unresolved_candidate_ids, "zero_valid_candidate_ids": zero_valid_candidate_ids, "failed_source_candidate_ids": failed_source_candidate_ids}
assert file_sha256(REVIEWED_DIR / "records.jsonl") == review["input_records_sha256"]
assert file_sha256(REVIEWED_DIR / "config.json") == review["reviewed_config_sha256"]
assert sum(review["totals"].values()) == 600 and len(review["candidates"]) == 60
print(json.dumps({"totals": review["totals"], "invalid_task_ids": review["invalid_task_ids"],
                  "unresolved_task_ids": review["unresolved_task_ids"], "zero_valid_candidate_ids": zero_valid_candidate_ids,
                  "reviewed_inputs": review["reviewed"]["inputs"], "input_records_sha256": review["input_records_sha256"],
                  "review_sha256": file_sha256(REVIEW_PATH)}, indent=2))


{
  "totals": {
    "valid": 593,
    "invalid": 7,
    "unresolved": 0
  },
  "invalid_task_ids": [
    "838",
    "966",
    "1553",
    "1958"
  ],
  "unresolved_task_ids": [],
  "zero_valid_candidate_ids": [],
  "reviewed_inputs": 593,
  "input_records_sha256": "a65ac044f568a59539b80537312f0ec6b5c8fb9f6cea85c21769608238db865e",
  "review_sha256": "a1f37bb7dc07f7e35a17814bd19a42fd3335f7534a451a733e19fff2edcca8d7"
}
